In [1]:
import pandas as pd
import numpy as np
import json, os

games = pd.read_parquet("../data/raw/games.parquet")
box = pd.read_parquet("../data/raw/boxscores.parquet")

# Attach date, season, matchup, and win/loss to each box score row
game_meta = games[["GAME_ID", "TEAM_ID", "GAME_DATE", "SEASON", "MATCHUP", "WL"]].drop_duplicates()
box = box.merge(game_meta, on=["GAME_ID", "TEAM_ID"], how="left")

box["GAME_DATE"] = pd.to_datetime(box["GAME_DATE"])
box["IS_HOME"] = ~box["MATCHUP"].str.contains("@", na=False)
box["MIN_NUM"] = pd.to_numeric(box["MIN"].astype(str).str.split(":").str[0], errors="coerce")

box = box[box["SEASON"].notna() & box["MIN_NUM"].notna()]
print(f"{len(box):,} player-game rows with metadata")
print(box["SEASON"].value_counts().to_string())

25,775 player-game rows with metadata
SEASON
2023-24    25775


In [2]:
def fmt(x, dec=1):
    return "n/a" if pd.isna(x) else f"{x:.{dec}f}"

def player_season_doc(g):
    p = g.iloc[0]
    n = len(g)
    home, away = g[g["IS_HOME"]], g[~g["IS_HOME"]]
    wins, losses = g[g["WL"] == "W"], g[g["WL"] == "L"]

    fg  = g["FGM"].sum() / g["FGA"].sum() * 100 if g["FGA"].sum() else np.nan
    fg3 = g["FG3M"].sum() / g["FG3A"].sum() * 100 if g["FG3A"].sum() else np.nan
    ft  = g["FTM"].sum() / g["FTA"].sum() * 100 if g["FTA"].sum() else np.nan

    text = (
        f"{p['PLAYER_NAME']}, {p['TEAM_ABBREVIATION']}, {p['SEASON']} regular season. "
        f"Across {n} games in this dataset, averaged {fmt(g['PTS'].mean())} points, "
        f"{fmt(g['REB'].mean())} rebounds, {fmt(g['AST'].mean())} assists, "
        f"{fmt(g['STL'].mean())} steals, and {fmt(g['BLK'].mean())} blocks "
        f"in {fmt(g['MIN_NUM'].mean())} minutes per game. "
        f"Shot {fmt(fg)}% from the field, {fmt(fg3)}% from three "
        f"({fmt(g['FG3A'].mean())} attempts per game), and {fmt(ft)}% from the free throw line. "
        f"Averaged {fmt(g['TO'].mean())} turnovers and {fmt(g['PF'].mean())} fouls. "
        f"Plus-minus averaged {fmt(g['PLUS_MINUS'].mean())}. "
        f"Season highs: {int(g['PTS'].max())} points, {int(g['REB'].max())} rebounds, "
        f"{int(g['AST'].max())} assists."
    )

    if len(home) >= 3 and len(away) >= 3:
        text += (f" At home ({len(home)} games) averaged {fmt(home['PTS'].mean())} points; "
                 f"on the road ({len(away)} games) averaged {fmt(away['PTS'].mean())} points.")
    if len(wins) >= 3 and len(losses) >= 3:
        text += (f" In wins ({len(wins)} games) averaged {fmt(wins['PTS'].mean())} points; "
                 f"in losses ({len(losses)} games) averaged {fmt(losses['PTS'].mean())} points.")

    return {
        "id": f"player-{p['PLAYER_ID']}-{p['SEASON']}",
        "type": "player_season",
        "player": p["PLAYER_NAME"],
        "team": p["TEAM_ABBREVIATION"],
        "season": p["SEASON"],
        "games": n,
        "text": text,
    }

counts = box.groupby(["PLAYER_ID", "SEASON"]).size()
keep = set(counts[counts >= 10].index)

player_docs = [
    player_season_doc(g)
    for key, g in box.groupby(["PLAYER_ID", "SEASON"])
    if key in keep
]
print(f"{len(player_docs)} player-season documents\n")
print(player_docs[0]["text"])

497 player-season documents

LeBron James, LAL, 2023-24 regular season. Across 69 games in this dataset, averaged 25.5 points, 7.3 rebounds, 8.3 assists, 1.2 steals, and 0.6 blocks in 34.9 minutes per game. Shot 53.9% from the field, 40.3% from three (5.1 attempts per game), and 74.4% from the free throw line. Averaged 3.5 turnovers and 1.1 fouls. Plus-minus averaged 2.5. Season highs: 40 points, 20 rebounds, 17 assists. At home (37 games) averaged 24.5 points; on the road (32 games) averaged 26.7 points. In wins (39 games) averaged 26.4 points; in losses (30 games) averaged 24.4 points.


In [3]:
games["GAME_DATE"] = pd.to_datetime(games["GAME_DATE"])
games["IS_HOME"] = ~games["MATCHUP"].str.contains("@", na=False)

def team_season_doc(g):
    t = g.iloc[0]
    n = len(g)
    wins = (g["WL"] == "W").sum()
    home, away = g[g["IS_HOME"]], g[~g["IS_HOME"]]

    text = (
        f"{t['TEAM_NAME']} ({t['TEAM_ABBREVIATION']}), {t['SEASON']} regular season. "
        f"Record: {wins}-{n - wins} across {n} games. "
        f"Averaged {fmt(g['PTS'].mean())} points, {fmt(g['REB'].mean())} rebounds, "
        f"{fmt(g['AST'].mean())} assists, and {fmt(g['TOV'].mean())} turnovers per game. "
        f"Shot {fmt(g['FGM'].sum()/g['FGA'].sum()*100)}% from the field and "
        f"{fmt(g['FG3M'].sum()/g['FG3A'].sum()*100)}% from three. "
        f"Average point differential: {fmt(g['PLUS_MINUS'].mean())}. "
        f"Home record: {(home['WL']=='W').sum()}-{(home['WL']=='L').sum()}. "
        f"Road record: {(away['WL']=='W').sum()}-{(away['WL']=='L').sum()}. "
        f"Highest-scoring game: {int(g['PTS'].max())} points. "
        f"Lowest-scoring game: {int(g['PTS'].min())} points."
    )
    return {
        "id": f"team-{t['TEAM_ID']}-{t['SEASON']}",
        "type": "team_season",
        "team": t["TEAM_NAME"],
        "team_abbr": t["TEAM_ABBREVIATION"],
        "season": t["SEASON"],
        "games": n,
        "text": text,
    }

team_docs = [team_season_doc(g) for _, g in games.groupby(["TEAM_ID", "SEASON"])]
print(f"{len(team_docs)} team-season documents\n")
print(team_docs[0]["text"])

60 team-season documents

Atlanta Hawks (ATL), 2023-24 regular season. Record: 36-46 across 82 games. Averaged 118.3 points, 44.7 rebounds, 26.6 assists, and 12.8 turnovers per game. Shot 46.5% from the field and 36.4% from three. Average point differential: -2.2. Home record: 21-20. Road record: 15-26. Highest-scoring game: 152 points. Lowest-scoring game: 95 points.


In [4]:
docs = player_docs + team_docs
os.makedirs("../data/corpus", exist_ok=True)

with open("../data/corpus/documents.jsonl", "w") as f:
    for d in docs:
        f.write(json.dumps(d) + "\n")

print(f"{len(docs)} documents saved")
print(f"Average length: {np.mean([len(d['text'].split()) for d in docs]):.0f} words")
print(f"Types: {pd.Series([d['type'] for d in docs]).value_counts().to_dict()}")

557 documents saved
Average length: 89 words
Types: {'player_season': 497, 'team_season': 60}
